In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm   # or multivariate_normal later if needed
from simulator import simulate

LOAD PRIOR CODE FROM VANILLA ABC

In [ ]:
infected_df = pd.read_csv("data/infected_timeseries.csv")
rewire_df = pd.read_csv("data/rewiring_timeseries.csv")
degree_df = pd.read_csv("data/final_degree_histograms.csv")

inf_wide = infected_df.pivot(index="time", columns="replicate_id", values="infected_fraction")
rew_wide = rewire_df.pivot(index="time", columns="replicate_id", values="rewire_count")

N_REPS = inf_wide.shape[1]
T = inf_wide.shape[0] - 1


      replicate_id  degree  count
0                0       0      0
1                0       1      0
2                0       2      0
3                0       3      0
4                0       4      4
...            ...     ...    ...
1235            39      26      0
1236            39      27      0
1237            39      28      0
1238            39      29      0
1239            39      30      0

[1240 rows x 3 columns]


In [5]:
# ── Compute_summary_stats ───

def compute_summary_stats(inf_wide, rew_wide, deg_df):
    """
    Compute the 7-statistic summary vector used for ABC.

    Statistics chosen after CV filtering and within/cross-source correlation analysis:
    Infection : peak_height, auc, growth_rate, time_to_peak
    Rewiring  : total_rewires, rew_inf_ratio
    Degree    : var_degree

    rew_inf_ratio = mean(peak_rewire_count_per_rep) / (mean_peak_infected_fraction * 200)
    This normalises rewiring intensity by epidemic size, capturing the rho/beta interaction.
    Verified comparable between observed and simulated data: s_obs sits at the 36th percentile
    of the prior-predictive distribution (confirmed in pilot analysis).
    """
    # Infection
    peak_height  = inf_wide.max(axis=0).mean()
    time_to_peak = inf_wide.idxmax(axis=0).mean()
    auc          = inf_wide.sum(axis=0).mean()

    def _growth(col):
        early = col.iloc[1:8].values
        early = np.where(early <= 0, 1e-6, early)
        return np.polyfit(np.arange(len(early)), np.log(early), 1)[0]

    growth_rate = inf_wide.apply(_growth, axis=0).mean()

    # Rewiring
    total_rewires = rew_wide.sum(axis=0).mean()
    peak_rewires  = rew_wide.max(axis=0).mean()
    rew_inf_ratio = peak_rewires / (peak_height * 200)
    

    # Degree
    def _var_degree(grp):
        degrees = grp["degree"].values
        counts  = grp["count"].values
        mean_d  = np.average(degrees, weights=counts)
        return np.average((degrees - mean_d)**2, weights=counts)

    var_degree = deg_df.groupby(
        "replicate_id", group_keys=False
    ).apply(lambda g: pd.Series({"v": _var_degree(g)}), include_groups=False)["v"].mean()

    return {
        "peak_height":   peak_height,
        "auc":           auc,
        "growth_rate":   growth_rate,
        "time_to_peak":  time_to_peak,
        "total_rewires": total_rewires,
        "rew_inf_ratio": rew_inf_ratio,
        "var_degree":    var_degree,
    }


# Compute s_obs 
s_obs = compute_summary_stats(inf_wide, rew_wide, degree_df)

print("Observed summary statistics (s_obs):")
for k, v in s_obs.items():
    print(f"  {k:<18}: {v:.5f}")

Observed summary statistics (s_obs):
  peak_height       : 0.65712
  auc               : 11.39900
  growth_rate       : 0.37520
  time_to_peak      : 8.75000
  total_rewires     : 545.15000
  rew_inf_ratio     : 0.74662
  var_degree        : 10.34102


In [6]:
PRIOR = {
    "beta":  (0.05, 0.50),
    "gamma": (0.02, 0.20),
    "rho":   (0.00, 0.80),
}

def sample_prior(rng):
    beta  = rng.uniform(*PRIOR["beta"])
    gamma = rng.uniform(*PRIOR["gamma"])
    rho   = rng.uniform(*PRIOR["rho"])
    return beta, gamma, rho

In [7]:
REPS_PER_DRAW = 5

def simulate_summary(beta, gamma, rho, n_reps, rng):
    """
    Run n_reps simulations at (beta, gamma, rho) and return the
    averaged 6-statistic summary vector.
    """
    inf_cols = {}
    rew_cols = {}
    deg_rows = []

    for rep in range(n_reps):
        inf_frac, rew_counts, deg_hist = simulate(
            beta=beta, gamma=gamma, rho=rho, rng=rng
        )
        inf_cols[rep] = inf_frac
        rew_cols[rep] = rew_counts
        for d, c in enumerate(deg_hist):
            deg_rows.append({"replicate_id": rep, "degree": d, "count": int(c)})

    inf_w = pd.DataFrame(inf_cols)
    rew_w = pd.DataFrame(rew_cols)
    deg_d = pd.DataFrame(deg_rows)

    return compute_summary_stats(inf_w, rew_w, deg_d)

In [8]:
# Distance Function 

STAT_NAMES = list(s_obs.keys())
# ['peak_height', 'auc', 'growth_rate', 'time_to_peak',
#  'total_rewires', 'rew_inf_ratio', 'var_degree']

def normalized_distance(s_sim, s_obs_dict, sigma):
    """
    Normalized Euclidean distance between two summary vectors.

    Each component (s_sim[k] - s_obs[k]) is divided by sigma[k],
    the prior-predictive standard deviation of statistic k.
    This ensures all 7 statistics contribute equally to the distance
    regardless of their original scale.

    Parameters
    ----------
    s_sim     : dict,  simulated summary statistics
    s_obs_dict: dict,  observed summary statistics (s_obs)
    sigma     : dict,  normalizing constants (prior-predictive std per statistic)
    """
    d_sq = 0.0
    for k in STAT_NAMES:
        if sigma[k] > 0:
            d_sq += ((s_sim[k] - s_obs_dict[k]) / sigma[k]) ** 2
    return np.sqrt(d_sq)

ABC-SMC

## ABC-SMC Run

Run your `abc_smc(...)` function in the next code cell or paste your existing implementation here.
The analysis section below assumes the run returns:

- `populations`: a list of dictionaries with keys `particles`, `weights`, `distances`, `epsilon`
- `all_distances`: a list of accepted distance arrays for each round

If you already executed the SMC run in another session, just rerun that cell before the analysis cells below.


In [ ]:
# Example placeholder:
# rng = np.random.default_rng(123)
# sigma = {k: pd.read_csv("pilot_results.csv")[f"s_{k}"].std() for k in STAT_NAMES}
# eps_schedule = [1.8, 1.4, 1.1, 0.9]
# populations, all_distances = abc_smc(
#     n_particles=100,
#     eps_schedule=eps_schedule,
#     n_reps=REPS_PER_DRAW,
#     sigma=sigma,
#     rng=rng,
# )


## Analysis

The cells below turn the final ABC-SMC population into a posterior summary and compare it with the rejection ABC baselines.


In [ ]:
pilot = pd.read_csv("pilot_results.csv")
sigma = {k: pilot[f"s_{k}"].std() for k in STAT_NAMES}

if "populations" not in globals():
    raise RuntimeError("Run the ABC-SMC sampler first so that `populations` exists.")

final_population = populations[-1]

smc_df = pd.DataFrame(final_population["particles"], columns=["beta", "gamma", "rho"])
smc_df["weight"] = np.asarray(final_population["weights"])
smc_df["distance"] = np.asarray(final_population["distances"])

smc_df.head()


In [ ]:
def weighted_quantile(values, quantiles, sample_weight):
    values = np.asarray(values)
    quantiles = np.asarray(quantiles)
    sample_weight = np.asarray(sample_weight)

    order = np.argsort(values)
    values = values[order]
    sample_weight = sample_weight[order]

    cumulative = np.cumsum(sample_weight) / sample_weight.sum()
    return np.interp(quantiles, cumulative, values)


def weighted_summary(df, params=("beta", "gamma", "rho"), weight_col="weight"):
    w = df[weight_col].to_numpy()
    w = w / w.sum()
    rows = []
    for param in params:
        vals = df[param].to_numpy()
        mean = np.average(vals, weights=w)
        var = np.average((vals - mean) ** 2, weights=w)
        q025, q50, q975 = weighted_quantile(vals, [0.025, 0.5, 0.975], w)
        rows.append(
            {
                "parameter": param,
                "mean": mean,
                "sd": np.sqrt(var),
                "median": q50,
                "q025": q025,
                "q975": q975,
            }
        )
    return pd.DataFrame(rows)


smc_summary = weighted_summary(smc_df)
smc_summary


In [ ]:
round_rows = []
for i, pop in enumerate(populations, start=1):
    w = np.asarray(pop["weights"])
    w = w / w.sum()
    distances = np.asarray(pop["distances"])
    ess = 1.0 / np.sum(w ** 2)
    round_rows.append(
        {
            "round": i,
            "epsilon": pop["epsilon"],
            "n_particles": len(pop["particles"]),
            "mean_distance": np.average(distances, weights=w),
            "median_distance": weighted_quantile(distances, [0.5], w)[0],
            "min_distance": distances.min(),
            "max_distance": distances.max(),
            "ess": ess,
        }
    )

round_df = pd.DataFrame(round_rows)
round_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(round_df["round"], round_df["epsilon"], marker="o")
axes[0].set_title("Tolerance schedule")
axes[0].set_xlabel("Round")
axes[0].set_ylabel("epsilon")

axes[1].plot(round_df["round"], round_df["mean_distance"], marker="o", label="weighted mean")
axes[1].plot(round_df["round"], round_df["median_distance"], marker="s", label="weighted median")
axes[1].set_title("Accepted distances by round")
axes[1].set_xlabel("Round")
axes[1].set_ylabel("distance")
axes[1].legend()

axes[2].plot(round_df["round"], round_df["ess"], marker="o")
axes[2].set_title("Effective sample size")
axes[2].set_xlabel("Round")
axes[2].set_ylabel("ESS")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

weights = smc_df["weight"].to_numpy()
for ax, param in zip(axes, ["beta", "gamma", "rho"]):
    ax.hist(smc_df[param], bins=25, weights=weights, density=True, alpha=0.75, color="#2F6B8A")
    ax.set_title(f"ABC-SMC posterior: {param}")
    ax.set_xlabel(param)
    ax.set_ylabel("density")

plt.tight_layout()
plt.show()


In [ ]:
rejection_confirm = pd.read_csv("abc_results_confirm.csv")
rejection_full = pd.read_csv("abc_results_full.csv")

comparison_rows = []
for param in ["beta", "gamma", "rho"]:
    smc_vals = smc_df[param].to_numpy()
    smc_w = smc_df["weight"].to_numpy()
    smc_mean = np.average(smc_vals, weights=smc_w)
    smc_sd = np.sqrt(np.average((smc_vals - smc_mean) ** 2, weights=smc_w))
    smc_q025, smc_q975 = weighted_quantile(smc_vals, [0.025, 0.975], smc_w)

    comparison_rows.append(
        {
            "parameter": param,
            "smc_mean": smc_mean,
            "smc_sd": smc_sd,
            "smc_q025": smc_q025,
            "smc_q975": smc_q975,
            "rej5_mean": rejection_confirm[param].mean(),
            "rej5_sd": rejection_confirm[param].std(),
            "rej1_mean": rejection_full[param].mean(),
            "rej1_sd": rejection_full[param].std(),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


In [ ]:
pairs = [("beta", "gamma"), ("beta", "rho"), ("gamma", "rho")]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (x_name, y_name) in zip(axes, pairs):
    ax.scatter(rejection_full[x_name], rejection_full[y_name], s=16, alpha=0.25, label="Rejection ABC (1%)")
    ax.scatter(
        smc_df[x_name],
        smc_df[y_name],
        s=40 + 200 * smc_df["weight"],
        alpha=0.7,
        label="ABC-SMC",
    )
    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.set_title(f"{x_name} vs {y_name}")
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
obs_summary = pd.Series(s_obs, name="observed")

def summarize_simulated_summaries(df, weight_col=None):
    summary_cols = [f"s_{k}" for k in STAT_NAMES if f"s_{k}" in df.columns]
    if len(summary_cols) == len(STAT_NAMES):
        if weight_col is None:
            values = {col[2:]: df[col].mean() for col in summary_cols}
        else:
            w = df[weight_col].to_numpy()
            values = {col[2:]: np.average(df[col], weights=w) for col in summary_cols}
        return pd.Series(values)
    return None


if all(f"s_{k}" in smc_df.columns for k in STAT_NAMES):
    smc_sim_summary = summarize_simulated_summaries(smc_df, weight_col="weight")
else:
    smc_sim_summary = None

rej5_summary = summarize_simulated_summaries(rejection_confirm)
rej1_summary = summarize_simulated_summaries(rejection_full)

summary_comparison = pd.DataFrame({"observed": obs_summary, "rej5": rej5_summary, "rej1": rej1_summary})
if smc_sim_summary is not None:
    summary_comparison["smc"] = smc_sim_summary

summary_comparison["rej1_minus_obs"] = summary_comparison["rej1"] - summary_comparison["observed"]
if smc_sim_summary is not None:
    summary_comparison["smc_minus_obs"] = summary_comparison["smc"] - summary_comparison["observed"]

summary_comparison


## Interpretation

Use the outputs above to make three points:

1. `round_df` should show the ABC-SMC populations getting more selective as epsilon decreases.
2. `comparison_df` shows whether the ABC-SMC posterior is similar to or tighter than the rejection ABC posterior.
3. The pairwise scatter plots are especially useful for the `beta-rho` tradeoff: if the SMC cloud is more concentrated along the same ridge, then SMC is refining the posterior even if the wall-clock runtime is longer.

A short write-up template:

“The ABC-SMC posterior concentrates progressively over the sequence of tolerances, with later populations achieving lower accepted distances to the observed summaries. Relative to vanilla rejection ABC, the final ABC-SMC population should be interpreted as a weighted posterior approximation rather than an equally weighted accepted sample. In this problem the main identifiability feature remains the `beta-rho` tradeoff, but ABC-SMC provides a more structured sequential refinement of that posterior region.” 
